In [ ]:
from __future__ import annotations

import argparse
import json
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, Tuple

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.ensemble import (
    GradientBoostingClassifier,
    GradientBoostingRegressor,
    RandomForestClassifier,
    RandomForestRegressor,
    StackingClassifier,
    StackingRegressor,
)
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.feature_selection import SelectKBest, f_classif, f_regression
from sklearn.impute import IterativeImputer, SimpleImputer
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    precision_score,
    r2_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import KFold, StratifiedKFold, cross_val_score, train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, PolynomialFeatures, StandardScaler
from sklearn.svm import SVC, SVR
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor


# =========================
# Dev2: Base AutoML Engine
# =========================
def load_csv(file):
    df = pd.read_csv(file)
    df.columns = [c.strip() for c in df.columns]
    return df


def split_features_target(df, target_col):
    X = df.drop(columns=[target_col])
    y = df[target_col]
    return X, y


def infer_column_types(X):
    num_cols = list(X.select_dtypes(include=["int64", "float64"]).columns)
    cat_cols = [c for c in X.columns if c not in num_cols]
    return num_cols, cat_cols


def detect_problem_type(y):
    if y.nunique() <= max(20, int(0.05 * len(y))):
        return "classification"
    return "regression"


def _classification_min_count(y):
    counts = y.value_counts()
    if counts.empty:
        return 0
    return int(counts.min())


def _safe_cv(problem_type, y, preferred_splits=3):
    if problem_type != "classification":
        n_splits = min(preferred_splits, len(y))
        n_splits = max(2, n_splits)
        return KFold(n_splits=n_splits, shuffle=True, random_state=42)

    min_count = _classification_min_count(y)
    if min_count >= 2:
        n_splits = min(preferred_splits, min_count, len(y))
        n_splits = max(2, n_splits)
        return StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

    n_splits = min(preferred_splits, len(y))
    n_splits = max(2, n_splits)
    return KFold(n_splits=n_splits, shuffle=True, random_state=42)


def _ensure_writeable_array(x):
    return np.array(x, copy=True)


def build_preprocessor(num_cols, cat_cols, use_iterative=True):
    if use_iterative:
        num_pipe = Pipeline(
            [
                ("writeable", FunctionTransformer(_ensure_writeable_array, validate=False)),
                ("imputer", IterativeImputer(initial_strategy="median")),
                ("scaler", StandardScaler()),
            ]
        )
    else:
        num_pipe = Pipeline(
            [
                ("writeable", FunctionTransformer(_ensure_writeable_array, validate=False)),
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]
        )

    cat_pipe = Pipeline(
        [
            ("writeable", FunctionTransformer(_ensure_writeable_array, validate=False)),
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore")),
        ]
    )

    return ColumnTransformer(
        [
            ("num", num_pipe, num_cols),
            ("cat", cat_pipe, cat_cols),
        ]
    )


def get_feature_selector(problem_type, X):
    k = min(10, X.shape[1])
    return SelectKBest(
        score_func=f_classif if problem_type == "classification" else f_regression,
        k=k,
    )


def build_feature_engineering(X):
    if X.shape[0] > 5000:
        return "passthrough"

    steps = []
    if X.shape[1] > 20:
        steps.append(("pca", PCA(n_components=min(20, X.shape[1]))))
    if X.shape[1] < 15:
        steps.append(("poly", PolynomialFeatures(degree=2, include_bias=False)))
    return Pipeline(steps) if steps else "passthrough"


def smart_model_selector(X, y, problem_type):
    n_samples, _ = X.shape
    models = {}

    imbalance = False
    if problem_type == "classification":
        dist = y.value_counts(normalize=True)
        imbalance = dist.min() < 0.2

    if problem_type == "classification":
        if n_samples < 1000:
            models["KNN"] = KNeighborsClassifier()
            models["NaiveBayes"] = GaussianNB()

        if n_samples < 10000:
            models["SVM"] = SVC()
            models["Logistic"] = LogisticRegression(max_iter=2000)
            models["DecisionTree"] = DecisionTreeClassifier()

        if n_samples >= 10000:
            models["RandomForest"] = RandomForestClassifier()
            models["GradientBoost"] = GradientBoostingClassifier()

        if imbalance:
            models["BalancedRF"] = RandomForestClassifier(class_weight="balanced")
    else:
        if n_samples < 1000:
            models["KNN"] = KNeighborsRegressor()

        if n_samples < 10000:
            models["SVR"] = SVR()
            models["Linear"] = LinearRegression()
            models["DecisionTree"] = DecisionTreeRegressor()

        if n_samples >= 10000:
            models["RandomForest"] = RandomForestRegressor()
            models["GradientBoost"] = GradientBoostingRegressor()

    print("Selected Models:", list(models.keys()))
    return models


def build_ensemble(models, problem_type):
    estimators = [(name, model) for name, model in models.items()]

    if problem_type == "classification":
        return StackingClassifier(
            estimators=estimators,
            final_estimator=LogisticRegression(max_iter=2000),
        )
    return StackingRegressor(
        estimators=estimators,
        final_estimator=LinearRegression(),
    )


def train_models_core(X, y, preprocessor, selector, problem_type):
    models = smart_model_selector(X, y, problem_type)
    models = dict(list(models.items())[:4])
    scores = {}

    if X.shape[0] > 5000:
        X_sample = X.sample(5000, random_state=42)
        y_sample = y.loc[X_sample.index]
    else:
        X_sample, y_sample = X, y

    cv = _safe_cv(problem_type, y_sample, preferred_splits=3)
    scoring = "f1_weighted" if problem_type == "classification" else "r2"

    for name, model in models.items():
        pipe = Pipeline(
            [
                ("pre", preprocessor),
                ("feat", selector),
                ("eng", build_feature_engineering(X)),
                ("model", model),
            ]
        )

        score = cross_val_score(pipe, X_sample, y_sample, cv=cv, scoring=scoring, n_jobs=-1).mean()
        scores[name] = {scoring: score}
        print(f"{name} ({scoring}): {score:.4f}")

    return scores


def final_training(X, y, preprocessor, selector, best_model, problem_type):
    stratify_target = None
    if problem_type == "classification" and _classification_min_count(y) >= 2:
        stratify_target = y

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42,
        stratify=stratify_target,
    )

    pipe = Pipeline(
        [
            ("pre", preprocessor),
            ("feat", selector),
            ("eng", build_feature_engineering(X)),
            ("model", best_model),
        ]
    )

    pipe.fit(X_train, y_train)
    return pipe, X_train, X_test, y_train, y_test


def evaluate(pipe, X_train, X_test, y_train, y_test, problem_type):
    y_pred_train = pipe.predict(X_train)
    y_pred_test = pipe.predict(X_test)

    roc = None
    if problem_type == "classification" and hasattr(pipe, "predict_proba"):
        try:
            proba = pipe.predict_proba(X_test)
            if len(np.unique(y_test)) == 2:
                roc = roc_auc_score(y_test, proba[:, 1])
            else:
                roc = roc_auc_score(y_test, proba, multi_class="ovr")
        except Exception:
            roc = None

    if problem_type == "classification":
        return {
            "train_acc": accuracy_score(y_train, y_pred_train),
            "test_acc": accuracy_score(y_test, y_pred_test),
            "f1": f1_score(y_test, y_pred_test, average="weighted", zero_division=0),
            "precision": precision_score(
                y_test, y_pred_test, average="weighted", zero_division=0
            ),
            "recall": recall_score(
                y_test, y_pred_test, average="weighted", zero_division=0
            ),
            "roc_auc": roc,
        }
    return {
        "train_r2": r2_score(y_train, y_pred_train),
        "test_r2": r2_score(y_test, y_pred_test),
        "mae": mean_absolute_error(y_test, y_pred_test),
        "rmse": mean_squared_error(y_test, y_pred_test, squared=False),
    }


def run_automl(file_path, target_col):
    print("AUTO ML DOCTOR STARTED\n")

    df = load_csv(file_path)
    X, y = split_features_target(df, target_col)
    num_cols, cat_cols = infer_column_types(X)

    problem_type = detect_problem_type(y)
    print("Problem Type:", problem_type)

    pre = build_preprocessor(num_cols, cat_cols, use_iterative=(X.shape[0] < 2000))
    selector = get_feature_selector(problem_type, X)

    scores = train_models_core(X, y, pre, selector, problem_type)

    if problem_type == "classification":
        sorted_scores = sorted(
            [(model_name, metrics["f1_weighted"]) for model_name, metrics in scores.items()],
            key=lambda item: item[1],
            reverse=True,
        )
    else:
        sorted_scores = sorted(
            [(model_name, metrics["r2"]) for model_name, metrics in scores.items()],
            key=lambda item: item[1],
            reverse=True,
        )

    top_models = [name for name, _ in sorted_scores[:4]]

    models_dict = smart_model_selector(X, y, problem_type)
    selected_models = {name: models_dict[name] for name in top_models if name in models_dict}

    use_ensemble = False
    if len(sorted_scores) > 1 and abs(sorted_scores[0][1] - sorted_scores[1][1]) < 0.02:
        use_ensemble = True

    if use_ensemble and len(selected_models) > 1:
        best_model = build_ensemble(selected_models, problem_type)
        print("\nAuto-Ensemble of Top Models:", top_models)
    else:
        best_model = list(selected_models.values())[0]
        print("\nBest Single Model:", top_models[0])

    model, X_train, X_test, y_train, y_test = final_training(
        X, y, pre, selector, best_model, problem_type
    )

    eval_metrics = evaluate(model, X_train, X_test, y_train, y_test, problem_type)
    print("\nEvaluation:", eval_metrics)

    return model, eval_metrics, scores


# =========================
# Dev3: Optimization Engine
# =========================
@dataclass
class ScoreBundle:
    train: float
    test: float
    metric_name: str


def evaluate_model(model, X_train, X_test, y_train, y_test, problem_type) -> ScoreBundle:
    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)

    if problem_type == "classification":
        train_score = accuracy_score(y_train, train_pred)
        test_score = accuracy_score(y_test, test_pred)
        return ScoreBundle(train=train_score, test=test_score, metric_name="accuracy")

    train_score = r2_score(y_train, train_pred)
    test_score = r2_score(y_test, test_pred)
    return ScoreBundle(train=train_score, test=test_score, metric_name="r2")


def detect_issues(train_score: float, test_score: float, problem_type: str, n_samples: int) -> str:
    base_gap = 0.08 if problem_type == "classification" else 0.12
    sample_adjustment = min(0.04, max(0.0, (2000 - n_samples) / 50000))
    overfit_gap = base_gap + sample_adjustment

    if (train_score - test_score) > overfit_gap:
        return "overfitting"

    if problem_type == "classification":
        low_perf_threshold = 0.70
    else:
        low_perf_threshold = 0.25

    if train_score < low_perf_threshold and test_score < low_perf_threshold:
        return "underfitting"

    return "good"


def check_imbalance(y_train) -> bool:
    ratio = y_train.value_counts(normalize=True)
    if len(ratio) <= 1:
        return False

    minority_ratio = ratio.min()
    majority_to_minority = ratio.max() / max(minority_ratio, 1e-9)

    return minority_ratio < 0.2 or majority_to_minority > 4.0


def _dynamic_forest_size(n_samples: int, n_features: int, issue: str) -> int:
    if issue == "underfitting":
        base = 300
    elif issue == "overfitting":
        base = 120
    else:
        base = 200

    size_factor = min(1.5, max(0.8, np.log1p(max(n_samples, 10)) / 6.0))
    feat_factor = min(1.4, max(0.8, np.log1p(max(n_features, 2)) / 4.0))

    n_estimators = int(base * size_factor * feat_factor)
    return int(np.clip(n_estimators, 80, 600))


def _dynamic_neighbors(n_samples: int) -> int:
    k = int(np.sqrt(max(n_samples, 5)))
    if k % 2 == 0:
        k += 1
    return int(np.clip(k, 3, 35))


def _safe_tree_depth(n_features: int, issue: str):
    if issue == "underfitting":
        return None
    if issue == "overfitting":
        return max(3, min(12, int(np.log2(max(n_features, 2)) * 1.8)))
    return max(4, min(16, int(np.log2(max(n_features, 2)) * 2.2)))


def _build_candidate_estimators(
    problem_type: str,
    issue: str,
    imbalance: bool,
    n_samples: int,
    n_features: int,
    random_state: int = 42,
) -> Dict[str, Any]:
    n_estimators = _dynamic_forest_size(n_samples, n_features, issue)
    k_neighbors = _dynamic_neighbors(n_samples)
    max_features_reg = max(0.33, min(0.8, np.sqrt(max(n_features, 1)) / max(n_features, 1)))

    if problem_type == "classification":
        class_weight = "balanced" if imbalance else None
        logistic_c = 0.7 if issue == "overfitting" else (2.0 if issue == "underfitting" else 1.0)
        svc_c = 0.8 if issue == "overfitting" else (2.0 if issue == "underfitting" else 1.0)

        candidates = {
            "Logistic": LogisticRegression(
                max_iter=4000,
                C=logistic_c,
                class_weight=class_weight,
                solver="lbfgs",
            ),
            "SVM": SVC(
                C=svc_c,
                kernel="rbf",
                gamma="scale",
                class_weight=class_weight,
                probability=True,
            ),
            "KNN": KNeighborsClassifier(
                n_neighbors=k_neighbors,
                weights="distance" if issue == "overfitting" else "uniform",
            ),
            "DecisionTree": DecisionTreeClassifier(
                max_depth=_safe_tree_depth(n_features, issue),
                min_samples_leaf=2 if issue == "overfitting" else 1,
                class_weight=class_weight,
                random_state=random_state,
            ),
            "RandomForest": RandomForestClassifier(
                n_estimators=n_estimators,
                max_depth=_safe_tree_depth(n_features, issue),
                min_samples_leaf=2 if issue == "overfitting" else 1,
                max_features="sqrt",
                class_weight=class_weight,
                random_state=random_state,
                n_jobs=-1,
            ),
            "GradientBoost": GradientBoostingClassifier(
                n_estimators=max(80, int(n_estimators * 0.6)),
                learning_rate=0.05 if issue == "overfitting" else 0.1,
                max_depth=2 if issue == "overfitting" else 3,
                random_state=random_state,
            ),
            "NaiveBayes": GaussianNB(
                var_smoothing=1e-8 if issue == "overfitting" else 1e-9
            ),
        }

        if imbalance:
            candidates["BalancedRF"] = RandomForestClassifier(
                n_estimators=n_estimators,
                max_depth=_safe_tree_depth(n_features, issue),
                min_samples_leaf=2 if issue == "overfitting" else 1,
                max_features="sqrt",
                class_weight="balanced",
                random_state=random_state,
                n_jobs=-1,
            )
        return candidates

    svr_c = 0.8 if issue == "overfitting" else (3.0 if issue == "underfitting" else 1.5)
    svr_epsilon = 0.2 if issue == "overfitting" else 0.1

    return {
        "Linear": LinearRegression(),
        "SVR": SVR(
            C=svr_c,
            epsilon=svr_epsilon,
            kernel="rbf",
            gamma="scale",
        ),
        "KNN": KNeighborsRegressor(
            n_neighbors=k_neighbors,
            weights="distance" if issue == "overfitting" else "uniform",
        ),
        "DecisionTree": DecisionTreeRegressor(
            max_depth=_safe_tree_depth(n_features, issue),
            min_samples_leaf=2 if issue == "overfitting" else 1,
            random_state=random_state,
        ),
        "RandomForest": RandomForestRegressor(
            n_estimators=n_estimators,
            max_depth=_safe_tree_depth(n_features, issue),
            min_samples_leaf=2 if issue == "overfitting" else 1,
            max_features=max_features_reg,
            random_state=random_state,
            n_jobs=-1,
        ),
        "GradientBoost": GradientBoostingRegressor(
            n_estimators=max(80, int(n_estimators * 0.6)),
            learning_rate=0.05 if issue == "overfitting" else 0.1,
            max_depth=2 if issue == "overfitting" else 3,
            random_state=random_state,
        ),
    }


def improve_model(
    problem_type: str,
    issue: str,
    imbalance: bool,
    n_samples: int,
    n_features: int,
    random_state: int = 42,
):
    return _build_candidate_estimators(
        problem_type=problem_type,
        issue=issue,
        imbalance=imbalance,
        n_samples=n_samples,
        n_features=n_features,
        random_state=random_state,
    )


def _fit_candidate_in_same_structure(original_model, candidate_estimator, X_train, y_train):
    if isinstance(original_model, Pipeline):
        optimized_pipeline = clone(original_model)
        optimized_pipeline.set_params(model=candidate_estimator)
        optimized_pipeline.fit(X_train, y_train)
        return optimized_pipeline

    candidate = clone(candidate_estimator)
    candidate.fit(X_train, y_train)
    return candidate


def optimize_model(
    model,
    X_train,
    X_test,
    y_train,
    y_test,
    problem_type: str,
    random_state: int = 42,
) -> Dict[str, Any]:
    before = evaluate_model(model, X_train, X_test, y_train, y_test, problem_type)
    issue = detect_issues(before.train, before.test, problem_type, n_samples=len(X_train))

    imbalance = False
    if problem_type == "classification":
        imbalance = check_imbalance(y_train)

    candidate_estimators = improve_model(
        problem_type=problem_type,
        issue=issue,
        imbalance=imbalance,
        n_samples=len(X_train),
        n_features=X_train.shape[1],
        random_state=random_state,
    )

    best_candidate_name = None
    best_candidate_estimator = None
    best_candidate_model = None
    best_candidate_scores = None
    candidate_scores = {}
    failed_candidates = {}

    for name, estimator in candidate_estimators.items():
        try:
            fitted_model = _fit_candidate_in_same_structure(
                original_model=model,
                candidate_estimator=estimator,
                X_train=X_train,
                y_train=y_train,
            )
            scores = evaluate_model(
                fitted_model, X_train, X_test, y_train, y_test, problem_type
            )
            candidate_scores[name] = {"train": scores.train, "test": scores.test}

            if best_candidate_scores is None or scores.test > best_candidate_scores.test:
                best_candidate_name = name
                best_candidate_estimator = estimator
                best_candidate_model = fitted_model
                best_candidate_scores = scores
        except Exception as exc:
            failed_candidates[name] = str(exc)

    if best_candidate_scores is None:
        after = before
        final_model = model
        final_train = before.train
        final_test = before.test
        selected_version = "original"
    else:
        after = best_candidate_scores
        if after.test >= before.test:
            final_model = best_candidate_model
            final_train = after.train
            final_test = after.test
            selected_version = "improved"
        else:
            final_model = model
            final_train = before.train
            final_test = before.test
            selected_version = "original"

    return {
        "metric": before.metric_name,
        "before_train": before.train,
        "before_test": before.test,
        "after_train": after.train,
        "after_test": after.test,
        "final_train": final_train,
        "final_test": final_test,
        "final_model": final_model,
        "issue_detected": issue,
        "imbalance_detected": imbalance,
        "selected_model_version": selected_version,
        "best_candidate_name": best_candidate_name,
        "candidate_model": best_candidate_estimator,
        "candidate_scores": candidate_scores,
        "failed_candidates": failed_candidates,
    }


# ==========================================
# Dev1: Integrated Pipeline (Dev1+Dev2+Dev3)
# ==========================================
def _convert_numpy(value: Any) -> Any:
    if isinstance(value, (np.floating, np.integer, np.bool_)):
        return value.item()
    if isinstance(value, np.ndarray):
        return value.tolist()
    return value


def _sanitize_dict(data: Dict[str, Any]) -> Dict[str, Any]:
    return {key: _convert_numpy(value) for key, value in data.items()}


def run_dev1_data_pipeline(df: pd.DataFrame, target_col: str) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    if target_col not in df.columns:
        raise ValueError(
            f"Target column '{target_col}' not found. Available columns: {list(df.columns)}"
        )

    original_rows = len(df)
    missing_target_rows = int(df[target_col].isna().sum())
    duplicate_rows = int(df.duplicated().sum())

    clean_df = df.copy()
    clean_df = clean_df.dropna(subset=[target_col])
    clean_df = clean_df.drop_duplicates()

    if clean_df.empty:
        raise ValueError("Dataset is empty after removing rows with missing target/duplicates.")

    report = {
        "original_rows": original_rows,
        "rows_after_cleaning": len(clean_df),
        "removed_missing_target_rows": missing_target_rows,
        "removed_duplicate_rows": duplicate_rows,
        "columns": list(clean_df.columns),
    }
    return clean_df, report


def _select_best_dev2_model(scores: Dict[str, Dict[str, float]], problem_type: str):
    if problem_type == "classification":
        ranked = sorted(
            ((name, metric["f1_weighted"]) for name, metric in scores.items()),
            key=lambda item: item[1],
            reverse=True,
        )
    else:
        ranked = sorted(
            ((name, metric["r2"]) for name, metric in scores.items()),
            key=lambda item: item[1],
            reverse=True,
        )

    if not ranked:
        raise RuntimeError("Dev2 could not produce model scores.")

    top_model_names = [name for name, _ in ranked[:4]]
    return ranked, top_model_names


def run_full_pipeline(file_path: str, target_col: str, random_state: int = 42) -> Dict[str, Any]:
    print("STAGE 1/3 - DEV1 DATA PIPELINE")
    raw_df = load_csv(file_path)
    clean_df, dev1_report = run_dev1_data_pipeline(raw_df, target_col)
    X, y = split_features_target(clean_df, target_col)
    X = X.copy(deep=True)
    y = y.copy(deep=True)
    print(f"Rows after cleaning: {dev1_report['rows_after_cleaning']}")

    print("\nSTAGE 2/3 - DEV2 AUTO ML DOCTOR")
    problem_type = detect_problem_type(y)
    num_cols, cat_cols = infer_column_types(X)
    preprocessor = build_preprocessor(num_cols, cat_cols, use_iterative=(X.shape[0] < 2000))
    selector = get_feature_selector(problem_type, X)
    cv_scores = train_models_core(X, y, preprocessor, selector, problem_type)

    ranked, top_model_names = _select_best_dev2_model(cv_scores, problem_type)
    available_models = smart_model_selector(X, y, problem_type)
    selected_models = {
        name: available_models[name] for name in top_model_names if name in available_models
    }

    if not selected_models:
        raise RuntimeError("No compatible model selected from Dev2 shortlist.")

    use_ensemble = False
    if len(ranked) > 1 and abs(ranked[0][1] - ranked[1][1]) < 0.02:
        use_ensemble = True

    if use_ensemble and len(selected_models) > 1:
        dev2_model = build_ensemble(selected_models, problem_type)
        dev2_choice = {"type": "ensemble", "members": list(selected_models.keys())}
    else:
        model_name = next(iter(selected_models))
        dev2_model = selected_models[model_name]
        dev2_choice = {"type": "single", "members": [model_name]}

    trained_model, X_train, X_test, y_train, y_test = final_training(
        X, y, preprocessor, selector, dev2_model, problem_type
    )
    baseline_metrics = evaluate(trained_model, X_train, X_test, y_train, y_test, problem_type)
    print(f"Problem type: {problem_type}")
    print(f"Dev2 model choice: {dev2_choice}")

    print("\nSTAGE 3/3 - DEV3 AUTO OPTIMIZATION")
    optimization = optimize_model(
        trained_model,
        X_train,
        X_test,
        y_train,
        y_test,
        problem_type,
        random_state=random_state,
    )
    final_model = optimization["final_model"]
    final_metrics = evaluate(final_model, X_train, X_test, y_train, y_test, problem_type)
    print(f"Issue detected by Dev3: {optimization['issue_detected']}")
    print(f"Selected version: {optimization['selected_model_version']}")

    return {
        "data_report": dev1_report,
        "problem_type": problem_type,
        "dev2": {
            "ranked_models": ranked,
            "choice": dev2_choice,
            "baseline_metrics": _sanitize_dict(baseline_metrics),
        },
        "dev3": {
            "metric": optimization["metric"],
            "issue_detected": optimization["issue_detected"],
            "imbalance_detected": _convert_numpy(optimization["imbalance_detected"]),
            "selected_model_version": optimization["selected_model_version"],
            "best_candidate_name": optimization["best_candidate_name"],
            "before_train": _convert_numpy(optimization["before_train"]),
            "before_test": _convert_numpy(optimization["before_test"]),
            "after_train": _convert_numpy(optimization["after_train"]),
            "after_test": _convert_numpy(optimization["after_test"]),
            "final_train": _convert_numpy(optimization["final_train"]),
            "final_test": _convert_numpy(optimization["final_test"]),
            "candidate_scores": _sanitize_dict(
                {
                    name: {
                        "train": _convert_numpy(score["train"]),
                        "test": _convert_numpy(score["test"]),
                    }
                    for name, score in optimization["candidate_scores"].items()
                }
            ),
            "failed_candidates": optimization["failed_candidates"],
            "final_metrics": _sanitize_dict(final_metrics),
        },
    }


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(
        description="Integrated Dev1 -> Dev2 -> Dev3 AutoML pipeline runner."
    )
    parser.add_argument("--file-path", required=True, help="Path to input CSV file.")
    parser.add_argument("--target-col", required=True, help="Target column name in CSV.")
    parser.add_argument(
        "--output-json",
        default="pipeline_report.json",
        help="Where to save the final pipeline report JSON.",
    )
    parser.add_argument(
        "--random-state", type=int, default=42, help="Random seed for reproducibility."
    )
    return parser.parse_args()


def main() -> None:
    args = parse_args()
    report = run_full_pipeline(
        file_path=args.file_path,
        target_col=args.target_col,
        random_state=args.random_state,
    )

    output_path = Path(args.output_json)
    output_path.write_text(json.dumps(report, indent=2), encoding="utf-8")
    print("\nPIPELINE COMPLETED")
    print(f"Report saved to: {output_path.resolve()}")
    print(f"Final metrics: {report['dev3']['final_metrics']}")


# Notebook usage example:
# report = run_full_pipeline(file_path="credit_card_frauds.csv", target_col="is_fraud", random_state=42)
# Path("pipeline_report.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
# report